# 57 — DinoBloom-S embeddings (exploratory robustness check of the encoder; RQ1, RQ3, RQ4)

Re-embeds the same cells as notebooks 03 (KU-Optofil), 15 (RV-PBS) and 37 (MLL23) with **DinoBloom-S**
(Koch et al., MICCAI 2024; weights: Zenodo record 10908163, `DinoBloom-S.pth`, Apache-2.0), a ViT-S/14 trained
self-supervised on public peripheral-blood and bone-marrow single-cell images. Everything else is identical to the
DINOv2 run: 224 px resize, ImageNet mean/std, fp16 autocast, the same cells in the same row order, and the same
patient-grouped 5-fold linear probe. Only the encoder changes, so `scripts/57_dinobloom_check.py` can attribute any
difference in costs or in InfoDPP-PAC's stopping to the encoder.

This is one robustness check, not an encoder comparison. Validity of the certification tests does not depend on the
encoder (it rests on the sampling design); only costs and the selector's behaviour can change.
Note for interpretation: DinoBloom's training data include AML images from the Munich Leukemia Laboratory (the same
laboratory as MLL23), so MLL23 is a smaller shift for this encoder than for DINOv2. KU-Optofil and RV-PBS were not in
its training data.

**Inputs in `MyDrive/certified-adaptive-pbs/`** (the same files the earlier notebooks used):
`dataset.zip` (KU-Optofil, Zenodo 10.5281/zenodo.17333317), `cells.csv` (from `scripts/02_make_folds.py`),
`external_crops.zip` (from `scripts/14_crop_rvpbs.py`), and `mll23_v1/mll23_cells.csv` (output of notebook 37; used
only to check that the MLL23 row order is unchanged). MLL23 images are downloaded again from Zenodo (about 8.6 GB).

**Runtime:** T4 GPU. KU-Optofil about 15 min, RV-PBS 1 min, MLL23 download 20–60 min plus 10 min.
Set `RUN_MLL23 = False` in cell 1 to skip MLL23.

**Output:** `MyDrive/certified-adaptive-pbs/dinobloom_v1/` → copy the folder to `results/dinobloom_v1/` in the
repository (only `config.json` is tracked in git).

In [ ]:
# 0. Setup and GPU check
import os, re, json, time, hashlib, platform, zipfile, requests
import numpy as np, pandas as pd, torch, torch.nn as nn
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
else: print('WARNING: no GPU. Runtime > Change runtime type > T4 GPU (CPU works, but takes hours)')

In [ ]:
# 1. Paths and options
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/certified-adaptive-pbs'
OUT = f'{D}/dinobloom_v1'; os.makedirs(OUT, exist_ok=True)
RUN_MLL23 = True
for f in ['dataset.zip', 'cells.csv', 'external_crops.zip']:
    assert os.path.exists(f'{D}/{f}'), f'missing {D}/{f}'
if RUN_MLL23:
    assert os.path.exists(f'{D}/mll23_v1/mll23_cells.csv'), 'missing mll23_v1/mll23_cells.csv (copy data/mll23/mll23_cells.csv there)'
print('inputs found')
def sha256(p, bs=1 << 22):
    h = hashlib.sha256()
    with open(p, 'rb') as fh:
        for b in iter(lambda: fh.read(bs), b''): h.update(b)
    return h.hexdigest()

In [ ]:
# 2. DinoBloom-S: DINOv2 ViT-S/14 architecture (no ImageNet weights) + DinoBloom weights.
#    Primary source: Zenodo record 10908163 (linked from github.com/marrlab/DinoBloom). Fallback: the lab's
#    Hugging Face copy (MarrLab/DinoBloom, pytorch_model_s.bin). The checkpoint must fill EVERY model parameter
#    and leave nothing over (asserted); the source and checksum are recorded in config.json.
IMG_SIZE, PATCH, DIM = 224, 14, 384
def fresh_model():
    m = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14', pretrained=False, trust_repo=True)
    m.pos_embed = nn.Parameter(torch.zeros(1, 1 + (IMG_SIZE // PATCH) ** 2, DIM))   # trained at 224 px: 257 tokens
    return m
def read_ckpt(path):
    try: ck = torch.load(path, map_location='cpu', weights_only=True)
    except Exception as e:                   # the training checkpoint may hold non-tensor entries
        print('weights_only load failed (', type(e).__name__, ') -> loading the official checkpoint with weights_only=False')
        ck = torch.load(path, map_location='cpu', weights_only=False)
    sd = ck
    for key in ('teacher', 'model', 'state_dict'):
        if isinstance(sd, dict) and key in sd and isinstance(sd[key], dict): sd = sd[key]; print('using checkpoint entry:', key); break
    sd = {k: v for k, v in sd.items() if not k.startswith(('dino_head', 'ibot_head'))}
    sd = {k[len('backbone.'):] if k.startswith('backbone.') else k: v for k, v in sd.items()}
    if any(re.match(r'^blocks\.\d+\.\d+\.', k) for k in sd):      # chunked blocks (training layout) -> flat layout
        sd = {re.sub(r'^blocks\.\d+\.(\d+)\.', r'blocks.\1.', k): v for k, v in sd.items()}
    return sd
def try_load(path):
    m = fresh_model(); miss, unexp = m.load_state_dict(read_ckpt(path), strict=False)
    print('missing keys:', miss, '| unexpected keys:', unexp)
    return m, list(miss), list(unexp)

WEIGHTS_URL = 'https://zenodo.org/records/10908163/files/DinoBloom-S.pth?download=1'
WPATH = '/content/DinoBloom-S.pth'
if not os.path.exists(WPATH):
    with requests.get(WEIGHTS_URL, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(WPATH, 'wb') as fh:
            for chunk in r.iter_content(1 << 22): fh.write(chunk)
model, missing, unexpected = try_load(WPATH); W_SOURCE = WEIGHTS_URL
if missing or unexpected:
    print('Zenodo checkpoint did not map cleanly -> trying the Hugging Face copy MarrLab/DinoBloom')
    !pip -q install huggingface_hub
    from huggingface_hub import hf_hub_download
    WPATH = hf_hub_download(repo_id='MarrLab/DinoBloom', filename='pytorch_model_s.bin')
    model, missing, unexpected = try_load(WPATH); W_SOURCE = 'huggingface.co/MarrLab/DinoBloom/pytorch_model_s.bin'
assert not missing and not unexpected, 'checkpoint does not match the ViT-S/14 architecture: stop and report'
W_SHA256 = sha256(WPATH); print('weights:', W_SOURCE, '| sha256 =', W_SHA256)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.eval().to(device)
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)            # ImageNet statistics, as for DINOv2
SEED = 20260919; torch.manual_seed(SEED); np.random.seed(SEED)
torch.backends.cudnn.benchmark = False; torch.backends.cudnn.deterministic = True
with torch.inference_mode():
    z = model(torch.zeros(2, 3, IMG_SIZE, IMG_SIZE, device=device))
print('output shape:', tuple(z.shape)); assert z.shape == (2, DIM)

In [ ]:
# 3. Shared embedding routine: same preprocessing as notebooks 03, 15 and 37 (only the encoder differs)
from PIL import Image
import torchvision.transforms.functional as TF
class Images(torch.utils.data.Dataset):
    def __init__(self, paths): self.paths = list(paths)
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        im = Image.open(self.paths[i]).convert('RGB')
        return TF.normalize(TF.to_tensor(TF.resize(im, [IMG_SIZE, IMG_SIZE], antialias=True)), MEAN, STD), i
def embed(paths, bs=128):
    dl = torch.utils.data.DataLoader(Images(paths), batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)
    E = np.zeros((len(dl.dataset), DIM), np.float32); seen = np.zeros(len(E), bool); t0 = time.time()
    with torch.inference_mode():
        for b, (x, idx) in enumerate(dl):
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(device == 'cuda')):
                E[idx.numpy()] = model(x.to(device, non_blocking=True)).float().cpu().numpy()
            seen[idx.numpy()] = True
            if b % 50 == 0: print(f'  batch {b}/{len(dl)}  {time.time() - t0:.0f}s', flush=True)
    assert seen.all() and np.isfinite(E).all()
    return E

In [ ]:
# 4. KU-Optofil: the 28,280 cells of cells.csv, in cells.csv row order
LOCAL = '/content/ku'
if not os.path.isdir(LOCAL) or not os.listdir(LOCAL):
    os.makedirs(LOCAL, exist_ok=True)
    !cp "{D}/dataset.zip" /content/dataset.zip
    !unzip -q -o /content/dataset.zip -d {LOCAL}
KU_ZIP_SHA256 = sha256('/content/dataset.zip')
print('zip matches the DINOv2 run' if KU_ZIP_SHA256 == '97fc42d9cd0e15299e9a4091e286a3f390ce97483fe291d821a3bf2fb858b0ea'
      else 'note: dataset.zip differs from the one used for the DINOv2 run (the file checks below still apply)')
# rel_path in cells.csv is relative to the dataset root, the folder that directly holds train/ val/ test/ (scripts/02)
roots = [r for r, subs, _ in os.walk(LOCAL) if {'train', 'val', 'test'} <= set(subs)]
assert roots, f'no folder with train/ val/ test/ under {LOCAL}'
KU_ROOT = min(roots, key=len); print('dataset root:', KU_ROOT)
cells = pd.read_csv(f'{D}/cells.csv', dtype={'patient_id': str})
cells['abs_path'] = KU_ROOT + '/' + cells['rel_path'].str.replace('\\', '/', regex=False)
found = cells.abs_path.map(os.path.exists)
assert found.all(), f'{(~found).sum()} images not found, e.g. {cells.abs_path[~found].iloc[0]}'
ref = f'{D}/embeddings_v1/cells_features.csv'      # output of notebook 03 (DINOv2), if it is still on Drive
if os.path.exists(ref):
    assert (pd.read_csv(ref).image_name.values == cells.image_name.values).all(), 'row order differs from the DINOv2 run'
    print('row order identical to the DINOv2 run')
print(len(cells), 'cells |', cells.patient_id.nunique(), 'patients |', int(cells.is_blast.sum()), 'blasts')
E_ku = embed(cells.abs_path); print('KU-Optofil embeddings', E_ku.shape)

In [ ]:
# 5. Patient-grouped 5-fold probe, exactly as notebook 03 cell 7 (the teacher / classifier scores)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
y = cells.is_blast.values; p_oof = np.zeros(len(cells))
for f in sorted(cells.fold.unique()):
    tr, te = cells.fold.values != f, cells.fold.values == f
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=0.1, class_weight='balanced'))
    clf.fit(E_ku[tr], y[tr]); p_oof[te] = clf.predict_proba(E_ku[te])[:, 1]
    print(f'fold {f}: test blasts={y[te].sum():4d}  AUC={roc_auc_score(y[te], p_oof[te]):.3f}')
AUC = roc_auc_score(y, p_oof); thr = np.quantile(p_oof[y == 0], 0.95)
print(f'pooled out-of-fold AUC = {AUC:.4f} (DINOv2 run: 0.9788) | sensitivity at 95% specificity = {(p_oof[y == 1] >= thr).mean():.3f}')
np.save(f'{OUT}/ku_embeddings_dinobloom_s.npy', E_ku.astype(np.float16))
cells[['image_name', 'patient_id', 'cell_type', 'is_blast', 'fold', 'provided_split']].assign(probe_p_oof=p_oof) \
    .to_csv(f'{OUT}/ku_cells_features.csv', index=False)
print('saved KU-Optofil outputs')

In [ ]:
# 6. RV-PBS crops, in rv_pbs_crops.csv row order (as notebook 15)
zipfile.ZipFile(f'{D}/external_crops.zip').extractall('/content/ext')
RV_ZIP_SHA256 = sha256(f'{D}/external_crops.zip')
rv = pd.read_csv('/content/ext/rv_pbs_crops.csv'); rv['path'] = '/content/ext/rv_pbs_crops/' + rv.crop_id + '.jpg'
assert rv.path.map(os.path.exists).all()
E_rv = embed(rv.path, bs=64); print('RV-PBS embeddings', E_rv.shape, '|', int(rv.is_blast.sum()), 'blasts')
np.save(f'{OUT}/rvpbs_embeddings_dinobloom_s.npy', E_rv.astype(np.float16))
rv.drop(columns=['path']).to_csv(f'{OUT}/rvpbs_cells.csv', index=False)

In [ ]:
# 7. MLL23 (optional): download as notebook 37, rebuild the index, check the row order against the DINOv2 run
mll = None
if RUN_MLL23:
    L = '/content/mll23'; os.makedirs(L, exist_ok=True)
    SKIP = {'promyelocyte_atypical.zip', 'smudge_cell.zip', 'normoblast.zip'}
    rec = requests.get('https://zenodo.org/api/records/14277609', timeout=60).json()
    for f in rec['files']:
        name = f['key']
        if name in SKIP or os.path.isdir(f'{L}/img/{name[:-4]}'): continue
        dst = f'{L}/{name}'; t0 = time.time()
        with requests.get(f['links']['self'], stream=True, timeout=600) as r:
            r.raise_for_status()
            with open(dst, 'wb') as fh:
                for chunk in r.iter_content(1 << 22): fh.write(chunk)
        if f.get('checksum', '').startswith('md5:'):
            h = hashlib.md5()
            with open(dst, 'rb') as fh:
                for chunk in iter(lambda: fh.read(1 << 22), b''): h.update(chunk)
            assert h.hexdigest() == f['checksum'][4:], f'checksum mismatch for {name}'
        zipfile.ZipFile(dst).extractall(f'{L}/img/{name[:-4]}'); os.remove(dst)
        print(name, round(f['size'] / 1e6, 1), 'MB', f'{time.time() - t0:.0f}s', flush=True)
    rows = []
    for cls in sorted(os.listdir(f'{L}/img')):
        for root, _, fs in os.walk(f'{L}/img/{cls}'):
            for fn in fs:
                if fn.lower().endswith(('.tif', '.tiff', '.png', '.jpg', '.jpeg')):
                    full = os.path.join(root, fn)
                    rows.append(dict(cls=cls, file=full, relpath=os.path.relpath(full, f'{L}/img/{cls}'), image=fn))
    mll = pd.DataFrame(rows).sort_values(['cls', 'relpath']).reset_index(drop=True)
    ref = pd.read_csv(f'{D}/mll23_v1/mll23_cells.csv')
    assert len(ref) == len(mll) and (ref.cls.values == mll.cls.values).all() and (ref.relpath.values == mll.relpath.values).all(), \
        'MLL23 row order differs from the DINOv2 run: stop and report'
    print(len(mll), 'MLL23 images, row order identical to the DINOv2 run')
    E_mll = embed(mll.file); np.save(f'{OUT}/mll23_embeddings_dinobloom_s.npy', E_mll.astype(np.float16))
    mll[['cls', 'relpath', 'image']].to_csv(f'{OUT}/mll23_cells.csv', index=False)

In [ ]:
# 8. Configuration record
config = dict(notebook='57_dinobloom_embeddings_colab.ipynb', created_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
              encoder='DinoBloom-S (ViT-S/14), Koch et al., MICCAI 2024', weights_source=W_SOURCE, weights_sha256=W_SHA256,
              load_check=dict(missing=list(missing), unexpected=list(unexpected)), img_size=IMG_SIZE, mean=list(MEAN), std=list(STD),
              dtype_saved='float16', dim=DIM, seed=SEED, ku=dict(n=int(len(cells)), zip_sha256=KU_ZIP_SHA256, probe_pooled_auc=float(AUC),
              probe_threshold_95spec=float(thr)), rvpbs=dict(n=int(len(rv)), zip_sha256=RV_ZIP_SHA256),
              mll23=None if mll is None else dict(n=int(len(mll)), zenodo='10.5281/zenodo.14277609', skipped=sorted(SKIP)),
              torch=torch.__version__, python=platform.python_version(),
              gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
              note='Identical cells, row order and preprocessing as embeddings_v1 (DINOv2); only the encoder differs. Exploratory.')
json.dump(config, open(f'{OUT}/config.json', 'w'), indent=1)
print(json.dumps(config, indent=1)); print('\nSaved to', OUT, '-> copy to results/dinobloom_v1/ in the repository')